# 📘 Pillow 실전 응용

Pillow를 활용한 실전 프로젝트입니다.
썸네일 생성, 워터마크, 이미지 포맷 변환, GIF 생성 등을 다룹니다.

**학습 목표:**
- 썸네일 일괄 생성
- 워터마크 삽입
- 이미지 포맷 변환 (PNG, JPEG, WebP)
- GIF 애니메이션 생성
- PIL ↔ NumPy 변환과 OpenCV 연동

## 1. 썸네일 일괄 생성

여러 이미지를 일괄적으로 썸네일 크기로 변환하는 작업입니다.
웹 갤러리, 이미지 목록 등에 활용합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  썸네일 일괄 생성                        │
# │  여러 이미지를 지정 크기로 축소            │
# └─────────────────────────────────────────┘

import tempfile, os, shutil

# 테스트 이미지 5장 생성
sizes = [(800, 600), (1200, 900), (640, 480), (1024, 768), (500, 400)]
colors = [(200, 100, 100), (100, 200, 100), (100, 100, 200),
          (200, 200, 100), (200, 100, 200)]

tmpdir = tempfile.mkdtemp()
images = []
for i, (size, color) in enumerate(zip(sizes, colors)):
    img = Image.new('RGB', size, color)
    draw = ImageDraw.Draw(img)
    draw.text((10, 10), f'Image {i+1}: {size[0]}x{size[1]}', fill='white')
    draw.rectangle([50, 50, size[0]-50, size[1]-50], outline='white', width=2)
    filename = f'photo_{i+1}.jpg'
    filepath = os.path.join(tmpdir, filename)
    img.save(filepath, 'JPEG', quality=90)
    images.append(filepath)
    print(f'생성: {filename} ({size[0]}x{size[1]})')

# 썸네일 생성
thumb_dir = os.path.join(tmpdir, 'thumbnails')
os.makedirs(thumb_dir, exist_ok=True)
THUMB_SIZE = (150, 150)

print(f'\n=== 썸네일 생성 (최대 {THUMB_SIZE}) ===')
for filepath in images:
    img = Image.open(filepath)
    original_size = img.size
    img.thumbnail(THUMB_SIZE)  # 비율 유지 축소
    thumb_name = 'thumb_' + os.path.basename(filepath).replace('.jpg', '.png')
    img.save(os.path.join(thumb_dir, thumb_name))
    print(f'  {original_size[0]}x{original_size[1]} -> {img.size[0]}x{img.size[1]} ({thumb_name})')

# 썸네일 시각화
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
thumb_files = sorted(os.listdir(thumb_dir))
for ax, tf in zip(axes, thumb_files):
    img = Image.open(os.path.join(thumb_dir, tf))
    ax.imshow(img)
    ax.set_title(f'{img.size}')
    ax.axis('off')
plt.suptitle('생성된 썸네일')
plt.tight_layout()
plt.show()

shutil.rmtree(tmpdir)
print('\n💡 thumbnail(): 비율을 유지하면서 최대 크기 이내로 축소')
print('💡 resize(): 비율과 관계없이 지정한 크기로 변경')

## 2. 워터마크 삽입

이미지에 텍스트나 로고 워터마크를 삽입해 저작권을 보호합니다.
RGBA 모드와 알파 채널을 활용해 투명도를 조절합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  워터마크 삽입                           │
# │  텍스트 워터마크 + 반복 워터마크          │
# └─────────────────────────────────────────┘

# 배경 이미지 (풍경)
img = Image.new('RGB', (600, 400), (120, 170, 220))
draw = ImageDraw.Draw(img)
for y in range(200):
    r = int(60 + 160 * y / 200)
    g = int(100 + 80 * y / 200)
    b = int(220 - 30 * y / 200)
    draw.line([(0, y), (600, y)], fill=(b, g, r))  # 하늘
for y in range(200, 400):
    draw.line([(0, y), (600, y)], fill=(50, 130, 50))  # 땅
draw.ellipse([430, 40, 510, 120], fill=(255, 230, 80))  # 태양

# 1. 단일 텍스트 워터마크
watermark_single = img.copy()
wm_layer = Image.new('RGBA', img.size, (0, 0, 0, 0))
wm_draw = ImageDraw.Draw(wm_layer)

font_path = 'C:/Windows/Fonts/malgun.ttf' if os.path.exists('C:/Windows/Fonts/malgun.ttf') else None
if font_path:
    font = ImageFont.truetype(font_path, 30)
    font_small = ImageFont.truetype(font_path, 14)
else:
    font = ImageFont.load_default()
    font_small = font

wm_draw.text((400, 350), 'WATERMARK', fill=(255, 255, 255, 100), font=font)
watermark_single = Image.alpha_composite(img.convert('RGBA'), wm_layer).convert('RGB')

# 2. 반복 워터마크 (타일)
watermark_tiled = img.copy()
wm_tiled = Image.new('RGBA', img.size, (0, 0, 0, 0))
wm_draw2 = ImageDraw.Draw(wm_tiled)
for y in range(0, 400, 80):
    for x in range(0, 600, 200):
        wm_draw2.text((x, y), 'SAMPLE', fill=(255, 255, 255, 40), font=font)
watermark_tiled = Image.alpha_composite(img.convert('RGBA'), wm_tiled).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img); axes[0].set_title('원본')
axes[1].imshow(watermark_single); axes[1].set_title('단일 워터마크')
axes[2].imshow(watermark_tiled); axes[2].set_title('반복 워터마크')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 RGBA에서 알파값(0~255)으로 투명도 조절')
print('💡 alpha_composite로 원본과 워터마크 합성')
print('💡 반복 워터마크: 저작권 보호에 효과적')

## 3. 이미지 포맷 변환과 최적화

Pillow는 다양한 이미지 포맷 간 변환과 압축 옵션을 지원합니다.
JPEG 품질, PNG 압축, WebP 포맷 등을 활용해 파일 크기를 최적화합니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  이미지 포맷 변환과 최적화               │
# │  JPEG, PNG, WebP 포맷 비교              │
# └─────────────────────────────────────────┘

import tempfile, os, shutil

# 테스트 이미지
img = Image.new('RGB', (800, 600), (150, 180, 220))
draw = ImageDraw.Draw(img)
rng = np.random.default_rng(42)
for _ in range(50):  # 랜덤 도형으로 복잡도 추가
    x1, y1 = rng.integers(0, 800), rng.integers(0, 600)
    x2, y2 = x1 + rng.integers(20, 100), y1 + rng.integers(20, 80)
    color = tuple(rng.integers(50, 255, 3))
    draw.rectangle([x1, y1, x2, y2], fill=color)

tmpdir = tempfile.mkdtemp()

# JPEG 품질별 저장
jpeg_files = []
for quality in [95, 75, 50, 25]:
    path = os.path.join(tmpdir, f'q{quality}.jpg')
    img.save(path, 'JPEG', quality=quality)
    size_kb = os.path.getsize(path) / 1024
    jpeg_files.append((quality, size_kb, path))

# PNG 저장
png_path = os.path.join(tmpdir, 'image.png')
img.save(png_path, 'PNG')
png_size = os.path.getsize(png_path) / 1024

# WebP 저장
webp_lossy = os.path.join(tmpdir, 'lossy.webp')
img.save(webp_lossy, 'WEBP', quality=80)
webp_size = os.path.getsize(webp_lossy) / 1024

webp_lossless = os.path.join(tmpdir, 'lossless.webp')
img.save(webp_lossless, 'WEBP', lossless=True)
webp_lossless_size = os.path.getsize(webp_lossless) / 1024

print('=== 포맷별 파일 크기 비교 ===')
print(f'PNG:               {png_size:>8.1f} KB')
for q, s, _ in jpeg_files:
    print(f'JPEG (quality={q:>2d}): {s:>8.1f} KB')
print(f'WebP (lossy 80):   {webp_size:>8.1f} KB')
print(f'WebP (lossless):   {webp_lossless_size:>8.1f} KB')

# JPEG 품질별 시각화
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img); axes[0].set_title(f'원본\nPNG {png_size:.0f}KB')
for ax, (q, s, p) in zip(axes[1:], jpeg_files):
    ax.imshow(Image.open(p))
    ax.set_title(f'JPEG q={q}\n{s:.0f}KB')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

shutil.rmtree(tmpdir)

print('\n💡 JPEG: quality 매개변수(1~100)로 압축률 조절')
print('💡 PNG: 무손실 압축, 파일 크기가 큼')
print('💡 WebP: JPEG 대비 25~35% 작은 크기, 품질 유지')
print('💡 포맷 변환: img.save("output.webp", "WEBP")')

## 4. PIL ↔ NumPy 변환과 OpenCV 연동

Pillow와 NumPy/OpenCV 간 변환은 이미지 처리 파이프라인에서 필수적입니다.

> 💡 **색상 순서 주의!**
> - PIL Image: RGB 순서
> - OpenCV (NumPy): BGR 순서

In [ ]:
# ┌─────────────────────────────────────────┐
# │  PIL <-> NumPy 변환과 OpenCV 연동         │
# │  이미지 처리 파이프라인에서 상호 변환      │
# └─────────────────────────────────────────┘

import cv2

# PIL Image 생성
pil_img = Image.new('RGB', (200, 200), (200, 100, 50))
draw = ImageDraw.Draw(pil_img)
draw.rectangle([40, 40, 160, 160], fill=(255, 0, 0))   # 빨강
draw.circle = draw.ellipse([80, 80, 150, 150], fill=(0, 255, 0))  # 초록

# === PIL -> NumPy ===
np_arr = np.array(pil_img)
print(f'PIL -> NumPy: shape={np_arr.shape}, dtype={np_arr.dtype}')
print(f'NumPy 픽셀 (0,0): {np_arr[0, 0]}  # RGB 순서')

# === NumPy -> PIL ===
pil_back = Image.fromarray(np_arr)
print(f'NumPy -> PIL: size={pil_back.size}, mode={pil_back.mode}')

# === PIL -> OpenCV (RGB -> BGR) ===
cv_arr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
print(f'\nPIL -> OpenCV: shape={cv_arr.shape}')
print(f'OpenCV 픽셀 (0,0): {cv_arr[0, 0]}  # BGR 순서')

# === OpenCV -> PIL (BGR -> RGB) ===
pil_from_cv = Image.fromarray(cv2.cvtColor(cv_arr, cv2.COLOR_BGR2RGB))
print(f'OpenCV -> PIL: size={pil_from_cv.size}, mode={pil_from_cv.mode}')

# 변환 파이프라인 예시
print('\n=== 변환 파이프라인 예시 ===')
print('1. PIL -> OpenCV: cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)')
print('2. OpenCV -> PIL: Image.fromarray(cv2.cvtColor(cv_arr, cv2.COLOR_BGR2RGB))')
print('3. PIL -> NumPy: np.array(pil_img)')
print('4. NumPy -> PIL: Image.fromarray(np_arr)')

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(pil_img); axes[0].set_title('PIL Image (RGB)')
axes[1].imshow(cv2.cvtColor(cv_arr, cv2.COLOR_BGR2RGB)); axes[1].set_title('OpenCV (BGR->RGB 표시)')
axes[2].imshow(pil_from_cv); axes[2].set_title('PIL 복원 (동일 확인)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 핵심: PIL은 RGB, OpenCV는 BGR! 변환 시 색상 순서에 주의')

## 🎯 연습 문제

1. 이미지 폴더의 모든 JPEG 파일을 읽어 150x150 썸네일로 저장하는 함수를 작성하세요.
2. 반투명 워터마크를 이미지 우측 하단에 배치하세요.
3. JPEG 품질을 10~95까지 5단계로 변경하며 저장하고, 파일 크기와 화질 차이를 비교하세요.
4. PIL 이미지를 OpenCV로 변환하여 Canny 엣지 검출을 적용한 후 다시 PIL로 변환하세요.
5. 여러 PNG 이미지를 불러와 하나의 모자이크 이미지(예: 3x3 그리드)로 합성하세요.